<a href="https://colab.research.google.com/github/Meru-dog/titanic-wandb-ml/blob/main/titanic_wanb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q wandb gradio

In [ ]:
import os
import wandb
from google.colab import userdata

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
wandb.login()

run = wandb.init(
    project="titanic-ml",
    name="setup-check",
    job_type="setup",
)
run.log({"setup_check": 1})
run.finish()

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame.copy()

print("Number of Rows and Columns:", df.shape)
print("Columns Name:", df.columns.tolist())
df.head()

In [ ]:
summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Number of missing values": df.isna().sum(),
    "Deficiency Rate(%)": (df.isna().mean() * 100).round(1),
})
summary

In [ ]:
print(df["survived"].value_counts(dropna=False))
print((df["survived"].value_counts(normalize=True) * 100).round(1))

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGET = "survived"
FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
EXCLUDED = ["boat", "body", "name", "ticket", "cabin", "home.dest"]

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

In [ ]:
assert not set(EXCLUDED) & set(X_train.columns)
assert not set(X_train.index) & set(X_test.index)

print("for training:", X_train.shape, f"Survival Rate: {y_train.mean():.1%}")
print("for testing:", X_test.shape, f"Survival Rate: {y_test.mean():.1%}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = ["age", "sibsp", "parch", "fare"]
CATEGORICAL_FEATURES = ["pclass", "sex", "embarked"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

In [ ]:
import numpy as np

X_train_processed = preprocessor.fit_transform(X_train)

print("processed shape:", X_train_processed.shape)
print("Number of missing values:", np.isnan(X_train_processed).sum())
print("Features Name:", preprocessor.get_feature_names_out().tolist())

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate

PROJECT_NAME = "titanic-ml"
METRICS = ["f1", "accuracy", "precision", "recall", "roc_auc"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

def run_cv_experiment(run_name, model):
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model),
    ])

    config = {
        "model_name": type(model).__name__,
        "model_params": model.get_params(),
        "dataset": "openml-titanic-v1",
        "features": FEATURES,
        "test_size": TEST_SIZE,
        "cv_folds": cv.get_n_splits(),
        "random_state": RANDOM_STATE,
        "notebook": "titanic_wandb.ipynb",
    }

    with wandb.init(
        project=PROJECT_NAME,
        name=run_name,
        job_type="cross-validation",
        config=config,
    ) as run:
        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=METRICS,
        )

        summary = {"run_name": run_name}
        for metric in METRICS:
            values = scores[f"test_{metric}"]
            summary[f"cv_{metric}_mean"] = values.mean()
            summary[f"cv_{metric}_std"] = values.std()

        run.summary.update(summary)

    results.append(summary)
    return summary

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

run_cv_experiment(
    "baseline-majority",
    DummyClassifier(strategy="most_frequent"),
)

run_cv_experiment(
    "baseline-logistic-regression",
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)

columns = [
    "run_name",
    "cv_f1_mean",
    "cv_f1_std",
    "cv_accuracy_mean",
    "cv_roc_auc_mean",
]
pd.DataFrame(results)[columns].round(3)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_settings = [
    {
        "run_name": "rf-unlimited-depth",
        "max_depth": None,
        "min_samples_leaf": 1,
    },
    {
        "run_name": "rf-depth-10-leaf-2",
        "max_depth": 10,
        "min_samples_leaf": 2,
    },
    {
        "run_name": "rf-depth-5-leaf-2",
        "max_depth": 5,
        "min_samples_leaf": 2,
    },
]

for setting in rf_settings:
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=setting["max_depth"],
        min_samples_leaf=setting["min_samples_leaf"],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    run_cv_experiment(setting["run_name"], model)

In [ ]:
results_df = (
    pd.DataFrame(results)
    .drop_duplicates("run_name", keep="last")
    .sort_values("cv_f1_mean", ascending=False)
)

results_df[[
    "run_name",
    "cv_f1_mean",
    "cv_f1_std",
    "cv_accuracy_mean",
    "cv_precision_mean",
    "cv_recall_mean",
    "cv_roc_auc_mean",
]].round(3)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

final_model_params = {
    "max_iter": 1000,
    "random_state": RANDOM_STATE,
}

final_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LogisticRegression(**final_model_params)),
])

final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
y_prob = final_pipeline.predict_proba(X_test)[:, 1]

test_metrics = {
    "test_f1": f1_score(y_test, y_pred),
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_precision": precision_score(y_test, y_pred),
    "test_recall": recall_score(y_test, y_pred),
    "test_roc_auc": roc_auc_score(y_test, y_prob),
}

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

with wandb.init(
    project=PROJECT_NAME,
    name="final-logistic-regression",
    job_type="final-evaluation",
    config={
        "model_name": "LogisticRegression",
        "model_params": final_model_params,
        "dataset": "openml-titanic-v1",
        "features": FEATURES,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "notebook": "titanic_wandb.ipynb",
    },
) as run:
    run.summary.update(test_metrics)
    run.summary.update({
        "test_tn": int(tn),
        "test_fp": int(fp),
        "test_fn": int(fn),
        "test_tp": int(tp),
    })
    run.log({
        "test_confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_test.tolist(),
            preds=y_pred.tolist(),
            class_names=["not_survived", "survived"],
        )
    })

display(pd.Series(test_metrics).round(3).to_frame("score"))
display(pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actual: Death", "Actual: Alive"],
    columns=["Predicted: Death", "Predicted: Alive"],
))

In [ ]:
import joblib
import sklearn

MODEL_PATH = "titanic_logistic_regression.joblib"
joblib.dump(final_pipeline, MODEL_PATH)

with wandb.init(
    project=PROJECT_NAME,
    name="save-final-model",
    job_type="model-registration",
) as run:
    artifact = wandb.Artifact(
        name="titanic-logistic-regression",
        type="model",
        description="Titanic survival model with preprocessing pipeline",
        metadata={
            "model_name": "LogisticRegression",
            "model_params": final_model_params,
            "features": FEATURES,
            "dataset": "openml-titanic-v1",
            "sklearn_version": sklearn.__version__,
            **{name: float(value) for name, value in test_metrics.items()},
        },
    )
    artifact.add_file(MODEL_PATH)
    run.log_artifact(artifact)

In [ ]:
loaded_pipeline = joblib.load(MODEL_PATH)
loaded_pred = loaded_pipeline.predict(X_test)

assert np.array_equal(loaded_pred, y_pred)
print("The predictions matched after loading.")

In [ ]:
import gradio as gr

demo_model = joblib.load(MODEL_PATH)

def predict_survival(pclass, sex, age, sibsp, parch, fare, embarked):
    input_df = pd.DataFrame([{
        "pclass": int(pclass),
        "sex": sex,
        "age": age,
        "sibsp": sibsp,
        "parch": parch,
        "fare": fare,
        "embarked": embarked,
    }])[FEATURES]

    survival_prob = float(demo_model.predict_proba(input_df)[0, 1])
    return {
        "Alive": survival_prob,
        "Death": 1 - survival_prob,
    }

demo = gr.Interface(
    fn=predict_survival,
    inputs=[
        gr.Radio([1, 2, 3], value=3, label="pclass"),
        gr.Radio(["male", "female"], value="male", label="sex"),
        gr.Number(value=30, minimum=0, maximum=100, label="age"),
        gr.Number(value=0, minimum=0, precision=0, label="Number of siblings or spouses traveling"),
        gr.Number(value=0, minimum=0, precision=0, label="Number of Parents and Children"),
        gr.Number(value=8, minimum=0, label="Fare"),
        gr.Dropdown(["C", "Q", "S"], value="S", label="embarked"),
    ],
    outputs=gr.Label(num_top_classes=2, label="Prediction Probability"),
    title="Titanic Survival Prediction Demo",
    description="This is a demonstration for educational purposes. It is not intended for use in actual decision-making.",
    examples=[
        [1, "female", 30, 0, 0, 100, "C"],
        [3, "male", 30, 0, 0, 8, "S"],
    ],
)

demo.launch(share=False)